# Convert Markdown to interleaved Parquet

This tutorial loads 1,000 Markdown documents from [PIN-14M](https://huggingface.co/datasets/m-a-p/PIN-14M), converts their text and image references into interleaved rows on Ray, materializes the image bytes while writing Parquet, and reads one output shard back for inspection.

Use Python 3.11–3.13 and install the required packages before running the notebook:

```bash
pip install "nemo_curator[interleaved_cpu]"
```

In [ ]:
from io import BytesIO

import pandas as pd
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import display as ipy_display
from PIL import Image

from nemo_curator.backends.ray_data import RayDataExecutor
from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.interleaved import MarkdownToInterleavedStage
from nemo_curator.stages.interleaved.io.readers.parquet import InterleavedParquetReaderStage
from nemo_curator.stages.interleaved.io.writers.tabular import InterleavedParquetWriterStage
from nemo_curator.tasks import DocumentBatch, FileGroupTask

NUM_ROWS = 1_000
OUTPUT_PATH = "pin14m_interleaved"

## Load and inspect the raw Markdown

Streaming limits the download to the rows used by this tutorial. The preview shows the source document ID and its unparsed Markdown.

In [ ]:
dataset = load_dataset("m-a-p/PIN-14M", "pin", split="train", streaming=True)
raw_df = pd.DataFrame(list(dataset.take(NUM_ROWS)))
ipy_display(raw_df[["id", "md"]].head(3))

## Convert and write

`MarkdownToInterleavedStage` preserves document metadata and emits text and lazy image-reference rows in reading order. Downloading `content_image.tar.gz` once avoids repeatedly scanning the large compressed archive over HTTP. `materialize_on_write=True` then resolves each image before Parquet is written; the default `on_materialize_error="error"` stops the run instead of silently writing missing image bytes.

In [ ]:
image_archive = hf_hub_download(
    repo_id="m-a-p/PIN-14M",
    repo_type="dataset",
    filename="data/DocLayNet/content_image.tar.gz",
)
document = DocumentBatch(
    dataset_name="PIN-14M",
    data=raw_df,
    _metadata={"source_files": [f"hf://datasets/m-a-p/PIN-14M#rows=0:{NUM_ROWS}"]},
)

pipeline = Pipeline(
    name="markdown_to_interleaved",
    stages=[
        MarkdownToInterleavedStage(image_source_uri=image_archive),
        InterleavedParquetWriterStage(
            path=OUTPUT_PATH,
            materialize_on_write=True,
            mode="overwrite",
        ),
    ],
)

with RayClient():
    written = pipeline.run(executor=RayDataExecutor(), initial_tasks=[document])

parquet_paths = [path for task in written for path in task.data]
ipy_display(pd.DataFrame({"parquet_path": parquet_paths}).head())

## Read and inspect the interleaved data

Read one shard back through Curator. `binary_bytes` confirms materialization, then the final loop displays the first three images.

In [ ]:
file_task = FileGroupTask(dataset_name="PIN-14M", data=[parquet_paths[0]], _metadata={})
interleaved = InterleavedParquetReaderStage().process(file_task)
interleaved_df = interleaved.to_pandas()
preview = interleaved_df.assign(
    binary_bytes=interleaved_df["binary_content"].map(
        lambda value: len(value) if isinstance(value, (bytes, bytearray, memoryview)) else None
    )
)
ipy_display(preview[["sample_id", "position", "modality", "text_content", "source_ref", "binary_bytes"]].head(20))

for payload in interleaved_df.loc[interleaved_df["modality"] == "image", "binary_content"].head(3):
    with Image.open(BytesIO(payload)) as image:
        image.thumbnail((640, 480))
        ipy_display(image)